In [20]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error,r2_score

df = pd.read_csv('AWE_1000_score.csv')

def count_sentences(text):
    if pd.isnull(text): return 0
    sentences = re.split(r'[.!?\n]+', str(text))
    return len([s for s in sentences if s.strip() != ""])

def count_phrases(text):
    if pd.isnull(text): return 0

    korean_connectives = r'그리고|하지만|그러나|그런데|그래서|그러므로|또한|게다가|왜냐하면|따라서|반면|즉|결국|한편'
    punctuations = r'[,，、]'

    pattern = f'{punctuations}|{korean_connectives}'
    phrases = re.split(pattern, str(text))
    return len([p for p in phrases if p.strip() != ""])

def count_words(text):
    if pd.isnull(text): return 0
    return len(str(text).split())

df['sentence'] = df['content'].apply(count_sentences)
df['phrase'] = df['content'].apply(count_phrases)
df['word_count'] = df['content'].apply(count_words)

y = df['score']

print("Preview of Feature Extraction results：")
print(df[['sentence', 'phrase', 'word_count', 'score']].head())

Preview of Feature Extraction results：
   sentence  phrase  word_count  score
0        17       3          64     80
1         9       2          23     70
2         6       2          28     50
3         8       1          25     60
4         3       1          11     20


In [21]:
# Model1 sentence
X1 = df[['sentence']]
y = df['score']
X_train1, X_test1, y_train1, y_test1 = train_test_split(X1, y, test_size=0.2, random_state=42)

model1 = LinearRegression()
model1.fit(X_train1, y_train1)
y_pred1 = model1.predict(X_test1)
mse1 = mean_squared_error(y_test1, y_pred1)
r2_1 = r2_score(y_test1, y_pred1)
print(f"Model 1 (sentence only)  MSE: {mse1:.4f},  R²: {r2_1:.4f}")

Model 1 (sentence only)  MSE: 322.5980,  R²: 0.0789


In [22]:
# Model2 sentence + phrase
X2 = df[['sentence', 'phrase']]
y2 = df['score']
X_train2, X_test2, y_train2, y_test2 = train_test_split(X2, y2, test_size=0.2, random_state=42)

model2 = LinearRegression()
model2.fit(X_train2, y_train2)
y_pred2 = model2.predict(X_test2)
mse2 = mean_squared_error(y_test2, y_pred2)
r2_2 = r2_score(y_test2, y_pred2)
print(f"Model 2 (sentence + phrase)  MSE: {mse2:.4f},  R²: {r2_2:.4f}")

Model 2 (sentence + phrase)  MSE: 317.3059,  R²: 0.0940


In [23]:
# Model3 sentence + phrase + word_count
X3 = df[['sentence', 'phrase', 'word_count']]
y3 = df['score']
X_train3, X_test3, y_train3, y_test3 = train_test_split(X3, y3, test_size=0.2, random_state=42)

model3 = LinearRegression()
model3.fit(X_train3, y_train3)
y_pred3 = model3.predict(X_test3)
mse3 = mean_squared_error(y_test3, y_pred3)
r2_3 = r2_score(y_test3, y_pred3)
print(f"Model 3 (sentence + phrase + word_count)  MSE: {mse3:.4f},  R²: {r2_3:.4f}")

Model 3 (sentence + phrase + word_count)  MSE: 308.0811,  R²: 0.1204


In [24]:
results_df = pd.DataFrame({
    'Actual Score': y_test1.values,
    'Predicted (Sentence Only)': y_pred1,
    'Predicted (Sentence + Phrase)': y_pred2,
    'Predicted (All Features)': y_pred3
})

print("\n Compare the prediction results of the models")
print(results_df.head(10))

print("\n Model Perfomance Summary")
summary = pd.DataFrame({
    'Model': ['Model 1 (sentence)', 'Model 2 (sentence+phrase)', 'Model 3 (all features)'],
    'MSE': [mse1, mse2, mse3],
    'R²': [r2_1, r2_2, r2_3]
})
print(summary)


 Compare the prediction results of the models
   Actual Score  Predicted (Sentence Only)  Predicted (Sentence + Phrase)  \
0            90                   73.94834                      73.289414   
1            90                   72.66886                      75.075932   
2            70                   71.38938                      71.071157   
3            90                   72.66886                      71.215070   
4            20                   62.43302                      62.342043   
5            70                   68.83042                      69.818116   
6            80                   67.55094                      66.778557   
7            90                   71.38938                      71.071157   
8            80                   70.10990                      70.927244   
9            40                   67.55094                      67.743772   

   Predicted (All Features)  
0                 74.586516  
1                 74.066206  
2              


#### Conclusion:
The results show that each additional feature gradually improves model performance.
Adding phrase count (Model 2) slightly reduced MSE and improved R², suggesting
that connective usage provides some additional information beyond sentence count.
Further adding word count (Model 3) achieved the best performance (MSE: 308.08,
R²: 0.12), indicating that essay length is a meaningful predictor of score.
However, the overall R² remains low (~0.12), suggesting that surface-level
features alone are insufficient for accurate score prediction.